In [1]:
from pyspark.sql import functions as f
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("WorkingTest") \
    .enableHiveSupport() \
    .getOrCreate()
print("Spark started successfully")

Spark started successfully


In [ ]:
raw_df = spark.read.csv('/DataSet/Bronze/yellow_tripdata_2019-01.csv', header=True, inferSchema=True)

In [ ]:
raw_df.show(5)

+---+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|_c0|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+---+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|  0|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5| 

In [ ]:
before_filter = raw_df.count()
print(before_filter)

7696617


In [ ]:
raw_df.describe()

DataFrame[summary: string, _c0: string, VendorID: string, passenger_count: string, trip_distance: string, RatecodeID: string, store_and_fwd_flag: string, PULocationID: string, DOLocationID: string, payment_type: string, fare_amount: string, extra: string, mta_tax: string, tip_amount: string, tolls_amount: string, improvement_surcharge: string, total_amount: string, congestion_surcharge: string, airport_fee: string]

In [ ]:
# Zero Value Columns
raw_df.filter(
    (f.col('trip_distance')<=0)&
    (f.col('fare_amount')<0)&
    (f.col('total_amount')<0)
    ).show()

+-----+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|  _c0|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+-----+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
| 2544|       2| 2019-01-01 00:45:57|  2019-01-01 00:46:07|            1.0|          0.0|       1.0|                 N|         234|         234|           4|       -2.5

In [ ]:
raw_df.select('store_and_fwd_flag').distinct().show()

+------------------+
|store_and_fwd_flag|
+------------------+
|                 Y|
|                 N|
+------------------+



In [ ]:
raw_df.columns

['_c0',
 'VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'airport_fee']

In [ ]:
silver_df.columns

['VendorID',
 'pickup_datetime',
 'dropoff_datetime',
 'pickup_location_id',
 'dropoff_location_id',
 'passenger_count',
 'trip_distance',
 'fare_amount',
 'total_amount',
 'payment_type',
 'RatecodeID',
 'store_and_fwd_flag']

In [ ]:
GoldDF.columns

['VendorID',
 'pickup_datetime',
 'dropoff_datetime',
 'pickup_location_id',
 'dropoff_location_id',
 'passenger_count',
 'trip_distance',
 'fare_amount',
 'total_amount',
 'payment_type',
 'RatecodeID',
 'store_and_fwd_flag',
 'pickup_hour',
 'pickup_day',
 'pickup_month',
 'trip_duration']

In [ ]:
raw_df.filter(f.col("trip_distance") <= 0).count()

55089

In [ ]:
df_test = raw_df.withColumn("pickup_hour", f.hour("tpep_pickup_datetime"))
df_test.groupBy("pickup_hour").count().show()

+-----------+------+
|pickup_hour| count|
+-----------+------+
|         12|401173|
|         22|369041|
|          1|149254|
|         13|404153|
|         16|420843|
|          6|178598|
|          3| 78086|
|         20|423156|
|          5| 75533|
|         19|475186|
|         15|452691|
|         17|468479|
|          9|365935|
|          4| 61424|
|          8|373742|
|         23|281941|
|          7|304858|
|         10|361390|
|         21|409901|
|         11|375441|
+-----------+------+
only showing top 20 rows


In [ ]:
df_test = df_test.filter(f.col("trip_distance") > 0)
df_test.show(5)

+---+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+
|_c0|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|pickup_hour|
+---+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+
|  0|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         

In [ ]:
spark.sql("SELECT DISTINCT payment_type FROM TempGoldDF ORDER BY payment_type").show()

+------------+
|payment_type|
+------------+
|           1|
|           2|
|           3|
|           4|
+------------+



In [10]:
spark.sql("SELECT DISTINCT VendorID FROM TempGoldDF ORDER BY VendorID").show()

+--------+
|VendorID|
+--------+
|       1|
|       2|
|       4|
+--------+



In [39]:
GoldDF.select('RatecodeID').distinct().orderBy(f.col('RatecodeID').asc()).show()

+----------+
|RatecodeID|
+----------+
|       1.0|
|       2.0|
|       3.0|
|       4.0|
|       5.0|
|       6.0|
|      99.0|
+----------+



In [229]:
spark.sql("""
SHOW TABLES IN nyc_taxitrip_db
""").show(7)

+---------------+--------------+-----------+
|      namespace|     tableName|isTemporary|
+---------------+--------------+-----------+
|nyc_taxitrip_db|      dim_date|      false|
|nyc_taxitrip_db|  dim_location|      false|
|nyc_taxitrip_db|   dim_payment|      false|
|nyc_taxitrip_db|  dim_ratecode|      false|
|nyc_taxitrip_db|    dim_vendor|      false|
|nyc_taxitrip_db|    fact_trips|      false|
|nyc_taxitrip_db|nyc_trip_table|      false|
+---------------+--------------+-----------+
only showing top 7 rows
